# 20 — Broadcast Reach Asymmetry Pilot

**Purpose.** Vary the broadcast reach of the two political agents (`reach_a` for pro-climate / Green-aligned, `reach_b` for anti-climate / Reform-aligned) and observe how an *asymmetric* communication environment shifts package-index trajectories away from the symmetric Run 5 baseline.

**Baseline.** This notebook does **not** run a symmetric (`reach_a=reach_b=1.0`) cell — Run 5 (`data/output/experiments/20260425_010615/`, NB 19) already provides that baseline. Section 11 loads Run 5 and overlays it on the asymmetric trajectory for direct comparison.

**Conditions (run one at a time).** Toggle `CONDITION` in Section 1 and execute the notebook end-to-end. Each condition writes to its own `CHECKPOINT_DIR` and its own timestamped `out_path`, so the two runs do not collide.

| Condition | `reach_a` (pro-climate) | `reach_b` (anti-climate) | Interpretation |
|---|---|---|---|
| `C1_reform_dominant` | 0.25 | 1.0 | Reform-aligned voice dominates the airwaves |
| `C3_green_dominant`  | 1.0  | 0.25 | Green-aligned voice dominates the airwaves |

All other configuration is **identical to Run 5**: N=30, 7 days, seed 43, package mode, GT-with-rationale anchor, Condition-B debias, dual-model (`gpt-5-mini` for messaging, `claude-sonnet-4-6` for surveys), alternating P-A/P-B day order. Only the audience size each political agent broadcasts to changes.

**Subsampling mechanism.** `apply_reach_subsample()` runs once at simulation start, after `assign_political_exposure()`, replacing each political agent's `connected_citizens` with a deterministic uniform subsample of size `floor(reach * |audience|)`. Per-citizen `political_exposure` labels are unchanged; peer messaging is unaffected. Subsample RNG is seeded by `random_seed` (agent A) and `random_seed + 1` (agent B), so both conditions are reproducible.

**Cost.** Reduced reach means fewer broadcast-reception LLM calls per day, so each asymmetric condition is **cheaper than Run 5** (~138 min wall-time). Expect roughly 100–115 min per condition.

In [ ]:
import os, sys, random, logging, time, shutil
from contextlib import contextmanager
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

from cag.io.survey import load
from cag.abm.agent import SurveyedCitizen, PoliticalAgent
from cag.abm.environment import SurveyedNation
from cag.abm.attributes.opinion import (
    ALL_CLIMATE_POLICIES,
    ClimatePolicyID,
    PACKAGE_SCOPE,
)
from cag.abm.sim import (
    run_simulation,
    save_results,
    save_result_plots,
    collect_ground_truth,
    collect_package_ground_truth,
    _load_checkpoint_meta,
)

from gabm.abm.attributes.gender import GenderMap, GenderID
from gabm.abm.attributes.politics import PoliticsID
from gabm.abm.democracy.election import ElectionID
from cag.abm.attributes.region import UKRegionMap, RegionID
from cag.abm.attributes.education import SurveyEducationMap, EducationID
from cag.abm.attributes.ethnicity import SurveyEthnicityMap, EthnicityID
from cag.abm.attributes.income import SurveyIncomeMap, IncomeID
from cag.abm.attributes.politics import SurveyPoliticsMap
from cag.abm.attributes.family import SurveyFamilyMap, FamilyID
from cag.abm.democracy.elections.ukge2019 import UKGE2019VoteMap, UKGE2019VoteID
from cag.abm.democracy.elections.brexit import BrexitVoteMap, BrexitVoteID
from cag.abm.attributes.narratives import (
    SelftranscMap, SelfenhMap, OpennessMap, ConformTradMap, SDOMap, EDOMap, RWAMap,
    rescale_1_6, rescale_1_7,
)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Timing harness ──────────────────────────────────────────────
TIMINGS = []

@contextmanager
def timed(label):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        dt = time.perf_counter() - t0
        TIMINGS.append((label, dt))
        print(f"[TIMER] {label}: {dt:.2f}s")

print("Imports OK")

## 1. Run Configuration

Pick one condition and execute the notebook. To run the other condition, restart the kernel, change `CONDITION`, and re-execute. Each condition has its own `CHECKPOINT_DIR`.

In [ ]:
# ── Pick the condition for THIS kernel session ──────────────────
# Allowed: "C1_reform_dominant" | "C3_green_dominant"
CONDITION = "C1_reform_dominant"

REACH_PRESETS = {
    "C1_reform_dominant": {"reach_a": 0.25, "reach_b": 1.0,
                            "label": "Reform-dominant (anti-climate broadcasts to all; pro-climate to 25%)"},
    "C3_green_dominant":  {"reach_a": 1.0,  "reach_b": 0.25,
                            "label": "Green-dominant (pro-climate broadcasts to all; anti-climate to 25%)"},
}
if CONDITION not in REACH_PRESETS:
    raise ValueError(f"CONDITION must be one of {list(REACH_PRESETS)}, got {CONDITION!r}")
REACH_A = REACH_PRESETS[CONDITION]["reach_a"]
REACH_B = REACH_PRESETS[CONDITION]["reach_b"]

# ── Headline knobs (must match Run 5 except for reach) ──────────
N_CITIZENS = 30
N_DAYS = 7
RANDOM_SEED = 43
year = 2026
P_INTRA, P_INTER = 0.15, 0.02

# Resume control. Flip to True after a kernel crash to continue from the last
# completed-day checkpoint of the SAME condition.
RESUME = False

CHECKPOINT_DIR = Path(f"../data/output/experiments/20_reach_{CONDITION}")

config = {
    "n_citizens": N_CITIZENS,
    "communication_mode": "package",
    "package_policies": list(ALL_CLIMATE_POLICIES),
    "day0_anchor": "ground_truth_with_rationale",
    "days": [
        {"phases": ["P-A", "P-B", "C"]},  # Day 1: pro-climate first
        {"phases": ["P-B", "P-A", "C"]},  # Day 2: anti-climate first
        {"phases": ["P-A", "P-B", "C"]},  # Day 3
        {"phases": ["P-B", "P-A", "C"]},  # Day 4
        {"phases": ["P-A", "P-B", "C"]},  # Day 5
        {"phases": ["P-B", "P-A", "C"]},  # Day 6
        {"phases": ["P-A", "P-B", "C"]},  # Day 7
    ],
    "k_peers_per_day": 3,
    "llm_model": "gpt-5-mini",
    "llm_provider": "openai",
    "llm_temperature": 0.5,
    "survey_model": "claude-sonnet-4-6",
    "survey_provider": "anthropic",
    "thinking": False,
    "debias": True,
    "p_intra": P_INTRA,
    "p_inter": P_INTER,
    "random_seed": RANDOM_SEED,
    # ── The asymmetry knobs (this notebook's whole point) ──
    "reach_a": REACH_A,
    "reach_b": REACH_B,
}

random.seed(RANDOM_SEED)

print(f"CONDITION = {CONDITION}")
print(f"  {REACH_PRESETS[CONDITION]['label']}")
print(f"  reach_a={REACH_A}  reach_b={REACH_B}")
print(f"N_CITIZENS={N_CITIZENS}, N_DAYS={N_DAYS}, seed={RANDOM_SEED}")
print(f"Messaging: {config['llm_model']} ({config['llm_provider']})")
print(f"Surveys:   {config['survey_model']} ({config['survey_provider']}) "
      f"thinking={config['thinking']} debias={config['debias']}")
print(f"Day 0 anchor: {config['day0_anchor']}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"RESUME = {RESUME}")

## 2. Build Sample Nation

In [ ]:
UKGE2019_ELECTION_ID = ElectionID(0)
BREXIT_REFERENDUM_ID = ElectionID(1)


def build_nation():
    sn = SurveyedNation(
        year=year, place="UK",
        gender_map=GenderMap(),
        region_map=UKRegionMap(),
        education_map=SurveyEducationMap(),
        ethnicity_map=SurveyEthnicityMap(),
        income_map=SurveyIncomeMap(),
        politics_map=SurveyPoliticsMap(),
        family_map=SurveyFamilyMap(),
        ukge2019_vote_map=UKGE2019VoteMap(UKGE2019_ELECTION_ID),
        brexit_vote_map=BrexitVoteMap(BREXIT_REFERENDUM_ID),
        selftransc_map=SelftranscMap, selfenh_map=SelfenhMap,
        openness_map=OpennessMap, conformtrad_map=ConformTradMap,
        sdo_map=SDOMap, edo_map=EDOMap, rwa_map=RWAMap,
    )
    data = load("../data/yougov_survey_data/YouGovProcessedData.csv")
    data = data.sample(n=N_CITIZENS, random_state=RANDOM_SEED).reset_index(drop=True)
    for i in range(len(data)):
        row = data.iloc[i]
        sc = SurveyedCitizen(
            agent_id=row.get('ID', None), environment=sn,
            year_of_birth=year - int(row.get('age', 0)),
            gender_id=GenderID.MALE if int(row.get('male_dummy', 0)) == 1 else GenderID.FEMALE,
            region_id=RegionID(int(row.get('tprofile_GOR', 0))),
            education_id=EducationID(int(row.get('profile_education_level', 0))),
            income_id=IncomeID(int(row.get('tprofile_gross_household', 0))),
            ethnicity_id=EthnicityID(int(row.get('ethnicity_R', 0))),
            family_id=FamilyID.PARENT if int(row.get('parent_dummy', 0)) == 1 else FamilyID.NOT_PARENT,
            ukge2019_vote_id=UKGE2019VoteID(int(row.get('Vote2019R', 0))),
            brexit_vote_id=BrexitVoteID(int(row.get('pastvote_EURef', 0))),
            politics_id=PoliticsID(int(row.get('Political_Left_Right', 0))),
            selftransc_id=rescale_1_6(int(row.get('Selftransc_Val', 0))),
            selfenh_id=rescale_1_6(int(row.get('Selfenh_Values', 0))),
            openness_id=rescale_1_6(int(row.get('Openness', 0))),
            conformtrad_id=rescale_1_6(int(row.get('ConformTrad', 0))),
            sdo_id=rescale_1_7(int(row.get('SDO', 0))),
            edo_id=rescale_1_7(int(row.get('EDO', 0))),
            rwa_id=rescale_1_7(int(row.get('RWA', 0))),
            original_survey_data=data.iloc[i],
        )
        sn.agents_active[sc.id] = sc
    return sn


with timed("build_nation"):
    sn = build_nation()
print(f"Citizens loaded: {len(sn.agents_active)}")

## 3. Audience Sizes Under This Condition

Diagnostic: confirm `apply_reach_subsample()` produces the audience counts we expect *before* spending wall-time on broadcasts. The full broadcast audiences come from `assign_political_exposure()`; subsampling then trims each side independently.

In [ ]:
sn.political_agent_a = PoliticalAgent("agent_a", "pro_climate")
sn.political_agent_b = PoliticalAgent("agent_b", "anti_climate")

with timed("audience diagnostic"):
    sn.assign_political_exposure()
    full_a = len(sn.political_agent_a.connected_citizens)
    full_b = len(sn.political_agent_b.connected_citizens)
    sn.apply_reach_subsample(reach_a=REACH_A, reach_b=REACH_B, seed=RANDOM_SEED)
    sub_a = len(sn.political_agent_a.connected_citizens)
    sub_b = len(sn.political_agent_b.connected_citizens)
    sn.create_network(p_intra=P_INTRA, p_inter=P_INTER, seed=RANDOM_SEED)
    sn.assign_network_blocks()

print(f"agent_a (pro-climate)  : {sub_a}/{full_a} citizens reached  (reach_a={REACH_A})")
print(f"agent_b (anti-climate) : {sub_b}/{full_b} citizens reached  (reach_b={REACH_B})")

degrees = [len(c.network_neighbors) for c in sn.agents_active.values()]
print(f"\nMean degree: {np.mean(degrees):.2f}")
print(f"Agents with 0 neighbours: {sum(1 for d in degrees if d == 0)} / {len(degrees)}")
print(f"Total edges: {sn.network.number_of_edges()}")

with timed("collect_ground_truth"):
    gt_df = collect_ground_truth(sn.agents_active.values())
    gt_package_df = collect_package_ground_truth(sn.agents_active.values())

print(f"\nGT package mean: {gt_package_df['ground_truth'].mean():+.2f}  (n={len(gt_package_df)})")

## 4. Run Simulation (with per-day checkpoints)

Identical machinery to NB 19. The kernel-crash recovery story is unchanged: flip `RESUME = True` in Section 1 (keeping the same `CONDITION`) and re-execute.

In [ ]:
if RESUME:
    if not (CHECKPOINT_DIR / "checkpoint_meta.json").exists():
        raise FileNotFoundError(
            f"RESUME=True but no checkpoint at {CHECKPOINT_DIR}. "
            "Either flip RESUME=False to start fresh, or restore a checkpoint dir."
        )
    meta = _load_checkpoint_meta(CHECKPOINT_DIR)
    print(f"Resuming from checkpoint: last_completed_day={meta['last_completed_day']}, "
          f"written_at={meta['written_at']}")
    sn = build_nation()
    sn.political_agent_a = PoliticalAgent("agent_a", "pro_climate")
    sn.political_agent_b = PoliticalAgent("agent_b", "anti_climate")
else:
    if CHECKPOINT_DIR.exists():
        print(f"Removing existing checkpoint dir: {CHECKPOINT_DIR}")
        shutil.rmtree(CHECKPOINT_DIR)

results = None
try:
    with timed("run_simulation (TOTAL)"):
        results = run_simulation(
            config, sn,
            checkpoint_dir=CHECKPOINT_DIR,
            resume=RESUME,
            checkpoint_every_day=True,
        )
    print(f"\nRun completed. Checkpoint dir: {CHECKPOINT_DIR}")
except KeyboardInterrupt:
    print(f"\n*** Interrupted. Partial checkpoint at: {CHECKPOINT_DIR} ***")
    print("    Set RESUME = True in Section 1 and re-execute to continue.")
    raise

## 5. Result Frames at a Glance

In [ ]:
df_traj = results["opinion_trajectories"]
df_package = results["package_index_trajectories"]
df_ref = results["reflections"]
df_reasoning = results["survey_reasoning"]
df_messages = results["messages"]

print(f"opinion_trajectories      : {len(df_traj):>6} rows  days={sorted(df_traj['day'].unique().tolist())}")
print(f"package_index_trajectories: {len(df_package):>6} rows")
print(f"reflections               : {len(df_ref):>6} rows")
print(f"survey_reasoning          : {len(df_reasoning):>6} rows")
print(f"messages                  : {len(df_messages):>6} rows")

# Sanity check: per-day broadcast counts should equal subsampled audience sizes.
if not df_messages.empty and "message_type" in df_messages.columns:
    bcast = df_messages[df_messages["message_type"] == "political_broadcast"]
    if not bcast.empty:
        per_phase = bcast.groupby(["day", "phase"]).size()
        print("\nBroadcast deliveries per (day, phase):")
        print(per_phase.to_string())

## 6. Day-0 Anchor Verification

In [ ]:
day0 = df_traj[df_traj["day"] == 0].copy()
day0["policy_id_str"] = day0["policy_id"].astype(str)
gt = gt_df.copy()
gt["policy_id_str"] = gt["policy_id"].astype(str)
merged = day0.merge(gt, on=["agent_id", "policy_id_str"], how="inner")
mismatches = merged[merged["numeric"] != merged["ground_truth"]]
assert mismatches.empty, f"Day 0 anchor mismatch on {len(mismatches)} rows:\n{mismatches.head()}"
print(f"PASS: Day 0 opinion == GT for all {len(merged)} (agent, policy) pairs")

day0_reasoning = df_reasoning[df_reasoning["day"] == 0]
expected = N_CITIZENS * len(ALL_CLIMATE_POLICIES)
print(f"Day 0 rationales stored: {len(day0_reasoning)} / expected {expected}")
assert len(day0_reasoning) == expected, "Missing Day 0 rationales"

## 7. Package Index Trajectory vs Ground Truth

In [ ]:
print("Mean package index by day:")
for day, val in df_package.groupby("day")["package_index"].mean().items():
    print(f"  Day {int(day)}: {val:+.2f}")
gt_pkg_mean = gt_package_df["ground_truth"].mean()
print(f"GT package mean: {gt_pkg_mean:+.2f}")

fig, ax = plt.subplots(figsize=(9, 5))
for aid, sub in df_package.groupby("agent_id"):
    sub = sub.sort_values("day")
    ax.plot(sub["day"], sub["package_index"], alpha=0.25, color="steelblue",
            linewidth=0.8, marker="o", markersize=2.5)
llm_mean = df_package.groupby("day")["package_index"].mean()
ax.plot(llm_mean.index, llm_mean.values, color="black", linewidth=2.4,
        marker="o", label=f"LLM mean ({CONDITION})")
ax.axhline(gt_pkg_mean, color="red", linestyle="--", linewidth=1.5,
           label=f"GT package mean ({gt_pkg_mean:+.2f})")
ax.set_xlabel("Day")
ax.set_ylabel("Package index (−3 to +3)")
ax.set_ylim(-3.5, 3.5)
ax.set_title(f"Pro-Climate Package Index — {CONDITION} (reach_a={REACH_A}, reach_b={REACH_B})")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 8. Per-Policy Trajectories

In [ ]:
POLICY_SHORT_NAMES = {
    str(ClimatePolicyID.RENEWABLE_ENERGY): "Renewable Energy",
    str(ClimatePolicyID.BAN_FOSSIL_FUEL): "Ban Fossil Fuels",
    str(ClimatePolicyID.BAN_PETROL_CARS): "Ban Petrol Cars",
    str(ClimatePolicyID.GREEN_HOUSING): "Green Housing",
    str(ClimatePolicyID.CARBON_TAX): "Carbon Tax",
    str(ClimatePolicyID.CLIMATE_COMPENSATION): "Climate Compensation",
}

policies_in_sim = sorted(df_traj["policy_id"].unique())
ncols = 3
nrows = (len(policies_in_sim) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False)

for i, pid in enumerate(policies_in_sim):
    ax = axes[i // ncols, i % ncols]
    pdf = df_traj[df_traj["policy_id"] == pid]
    for aid, sub in pdf.groupby("agent_id"):
        sub = sub.sort_values("day")
        ax.plot(sub["day"], sub["numeric"], alpha=0.2, color="steelblue",
                linewidth=0.7, marker="o", markersize=2)
    llm_mean = pdf.groupby("day")["numeric"].mean()
    ax.plot(llm_mean.index, llm_mean.values, color="black", linewidth=2,
            marker="o", label="LLM mean")
    gt_p = gt_df[gt_df["policy_id"].astype(str) == str(pid)]
    if not gt_p.empty:
        gt_mean = gt_p["ground_truth"].mean()
        ax.axhline(gt_mean, color="red", linestyle="--", linewidth=1.3,
                   label=f"GT mean ({gt_mean:+.1f})")
    ax.set_xlabel("Day")
    ax.set_ylabel("Opinion")
    ax.set_ylim(-3.5, 3.5)
    ax.set_title(POLICY_SHORT_NAMES.get(str(pid), str(pid))[:40])
    ax.legend(fontsize=8)

for j in range(len(policies_in_sim), nrows * ncols):
    axes[j // ncols, j % ncols].set_visible(False)

fig.suptitle(f"Per-Policy Trajectories — {CONDITION}, N={N_CITIZENS}, {N_DAYS} days", fontsize=13)
plt.tight_layout()
plt.show()

## 9. Activity Counts + Sample Outputs

In [ ]:
print("Reflections per phase:")
print(df_ref.groupby("phase").size().to_string())
print(f"\nMessages logged: {len(df_messages)}")

print("\n--- Sample reflection per phase ---")
for phase, grp in df_ref.groupby("phase"):
    if grp.empty:
        continue
    row = grp.iloc[0]
    text = row.get("reflection") or row.get("text") or ""
    print(f"\n[{phase}] agent={row['agent_id']} day={row['day']}")
    print(text[:400])

print("\n--- Sample Day>=1 debias reasoning ---")
late = df_reasoning[df_reasoning["day"] > 0]
if late.empty:
    print("(no Day>=1 reasoning rows)")
else:
    row = late.iloc[0]
    print(f"agent={row['agent_id']} day={row['day']} policy={row['policy_id']}")
    print(row["reasoning"][:500])

## 10. Save Final Results + Plots + Timings

In [ ]:
out_path = save_results(results, output_dir="../data/output/experiments")
plot_paths = save_result_plots(results, out_path)

timings_df = pd.DataFrame(TIMINGS, columns=["section", "seconds"])
total = timings_df["seconds"].sum()
timings_df.to_csv(out_path / "timings.csv", index=False)

# Drop a small marker so the output dir is self-describing.
(out_path / "CONDITION.txt").write_text(
    f"{CONDITION}\nreach_a={REACH_A}\nreach_b={REACH_B}\n"
    f"baseline=Run 5 (data/output/experiments/20260425_010615)\n"
)

print(f"\nResults: {out_path}")
print(f"Plots:   {len(plot_paths)} files")
print(f"\n=== Wall-time summary ({total:.1f}s = {total/60:.1f} min) ===")
print(timings_df.to_string(index=False))
print(f"\nTimings written to {out_path / 'timings.csv'}")
print(f"\nCheckpoint dir ({CHECKPOINT_DIR}) can now be deleted if desired.")

## 11. Compare Against Run 5 (Symmetric Baseline)

Overlay this asymmetric condition's package-index mean trajectory on Run 5 (`reach_a = reach_b = 1.0`, otherwise identical config). The gap quantifies how much asymmetric reach moves the population away from the symmetric outcome.

Set `BASELINE_DIR` if you have re-run Run 5 to a different timestamp.

In [ ]:
BASELINE_DIR = Path("../data/output/experiments/20260425_010615")  # Run 5

if not BASELINE_DIR.exists():
    print(f"Baseline dir not found: {BASELINE_DIR} — skipping overlay.")
else:
    base_pkg = pd.read_csv(BASELINE_DIR / "package_index_trajectories.csv")
    base_mean = base_pkg.groupby("day")["package_index"].mean()
    asym_mean = df_package.groupby("day")["package_index"].mean()

    cmp_df = pd.DataFrame({
        "baseline (Run 5, symmetric)": base_mean,
        f"{CONDITION} (reach_a={REACH_A}, reach_b={REACH_B})": asym_mean,
    })
    cmp_df["delta"] = cmp_df.iloc[:, 1] - cmp_df.iloc[:, 0]
    print("Mean package index by day (asymmetric − baseline):")
    print(cmp_df.round(3).to_string())

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(base_mean.index, base_mean.values, color="gray", linewidth=2.2,
            marker="s", linestyle="--", label="Run 5 baseline (symmetric)")
    ax.plot(asym_mean.index, asym_mean.values, color="black", linewidth=2.4,
            marker="o", label=f"{CONDITION}")
    ax.axhline(gt_pkg_mean, color="red", linestyle=":", linewidth=1.2,
               label=f"GT package mean ({gt_pkg_mean:+.2f})")
    ax.set_xlabel("Day")
    ax.set_ylabel("Mean package index")
    ax.set_ylim(-3.5, 3.5)
    ax.set_title(f"Asymmetric reach vs symmetric baseline — {CONDITION}")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()